<a href="https://colab.research.google.com/github/wanchenlang-max/econ5200-lab/blob/main/lab_ch22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 22: Clustering Economies — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 40 min Core + 20 min Extension

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Topics:** K-Means clustering, feature standardization, PCA visualization, silhouette evaluation, UMAP comparison, reusable Python modules.

**Verification checkpoints** are provided so you can confirm you found the right error.

**Time estimate:** ~60 minutes

---

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Install required packages and import libraries
# -----------------------------------------------------------
!pip install wbgapi scikit-learn matplotlib seaborn umap-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import wbgapi as wb
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

np.random.seed(42)

print('Libraries loaded. Ready to diagnose.')

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Load WDI data (shared setup for all parts)
# -----------------------------------------------------------

indicators = {
    'NY.GDP.PCAP.PP.CD': 'gdp_per_capita_ppp',
    'SP.DYN.LE00.IN': 'life_expectancy',
    'SP.DYN.IMRT.IN': 'infant_mortality',
    'SE.PRM.ENRR': 'primary_enrollment',
    'SI.POV.GINI': 'gini_index',
    'EN.ATM.CO2E.PC': 'co2_per_capita',
    'IT.NET.USER.ZS': 'internet_users_pct',
    'NE.TRD.GNFS.ZS': 'trade_pct_gdp',
    'SL.UEM.TOTL.ZS': 'unemployment_rate',
    'SP.URB.TOTL.IN.ZS': 'urban_population_pct'
}

feature_names = list(indicators.values())

df = wb.data.DataFrame(list(indicators.keys()), mrv=1, labels=True)
df = df.rename(columns=indicators)
df = df.dropna(thresh=7)
df = df.fillna(df.median())

print(f'Countries retained: {len(df)}')
print(f'Features: {feature_names}')

---

## Part 1: DIAGNOSE — Find 4 Errors in This Clustering Pipeline

The code below attempts to cluster World Bank economies using K-Means.
There are **four deliberate errors** spread across four code cells:

1. A **preprocessing omission** error
2. An **API parameter** error
3. A **method ordering** error
4. A **reproducibility** error

**Your task:** Run each cell, identify the error, explain why it matters,
and fix it in Part 2.

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find and fix it.
# Step 1: Cluster raw (unstandardized) features
# -----------------------------------------------------------

# ERROR 1: Forgot to standardize before K-Means!
# GDP per capita ranges 300-120,000 while Gini ranges 25-65.
# Without StandardScaler, K-Means clusters almost entirely on GDP.

X_raw = df[feature_names].values  # Using RAW features — no scaling!

kmeans_bad = KMeans(n_clusters=4, init='k-means++', n_init='auto', random_state=42)
labels_bad = kmeans_bad.fit_predict(X_raw)

print('=== Clustering on RAW (unstandardized) features ===')
print(f'Cluster sizes: {np.bincount(labels_bad)}')
print()
for k in range(4):
    mask = labels_bad == k
    print(f'Cluster {k}: {mask.sum()} countries | '
          f'GDP/cap ${df.loc[mask, "gdp_per_capita_ppp"].mean():,.0f} | '
          f'Life Exp {df.loc[mask, "life_expectancy"].mean():.1f}')

print()
print('Notice: clusters are separated ONLY by GDP per capita.')
print('The other 9 features contribute almost nothing to the distance.')

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find and fix it.
# Step 2: Wrong argument name for n_clusters
# -----------------------------------------------------------

# ERROR 2: Using k=4 instead of n_clusters=4
# scikit-learn uses n_clusters, not k. This will raise a TypeError.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_names])

try:
    kmeans_wrong = KMeans(k=4, init='k-means++', random_state=42)
    kmeans_wrong.fit(X_scaled)
except TypeError as e:
    print(f'ERROR: {e}')
    print()
    print('The correct parameter name is n_clusters, not k.')
    print('scikit-learn\'s KMeans uses: KMeans(n_clusters=4)')

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find and fix it.
# Step 3: PCA applied BEFORE standardization
# -----------------------------------------------------------

# ERROR 3: PCA is applied to raw data, then results are standardized.
# This is backwards! PCA should be applied AFTER standardization.
# PCA finds directions of maximum variance — on raw data, the first PC
# will be dominated by the highest-scale feature (GDP per capita).

# Wrong order: PCA first, then scale
pca_wrong = PCA(n_components=2)
X_pca_wrong = pca_wrong.fit_transform(df[feature_names].values)  # Raw data!
X_pca_then_scaled = StandardScaler().fit_transform(X_pca_wrong)  # Scaling after PCA

print('PCA on RAW data:')
print(f'  PC1 explains {pca_wrong.explained_variance_ratio_[0]:.1%} of variance')
print(f'  PC2 explains {pca_wrong.explained_variance_ratio_[1]:.1%} of variance')
print()
print('PC1 loading vector (top 3):')
loadings = pd.Series(pca_wrong.components_[0], index=feature_names)
top_loadings = loadings.abs().nlargest(3)
for feat in top_loadings.index:
    print(f'  {feat}: {loadings[feat]:.4f}')
print()
print('Notice: PC1 is almost entirely GDP per capita.')
print('Standardize FIRST, then apply PCA.')

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find and fix it.
# Step 4: Missing random_state
# -----------------------------------------------------------

# ERROR 4: No random_state set — results change every time you run!
# K-Means uses random initialization. Without random_state,
# different runs may converge to different local minima.

X_scaled_ok = StandardScaler().fit_transform(df[feature_names])

# Run K-Means 3 times without random_state
results = []
for trial in range(3):
    km = KMeans(n_clusters=4, init='k-means++', n_init=1)  # No random_state!
    labels = km.fit_predict(X_scaled_ok)
    inertia = km.inertia_
    results.append((labels, inertia))
    print(f'Trial {trial+1}: WCSS = {inertia:.2f}, Cluster sizes = {np.bincount(labels)}')

# Check if results are identical
same_01 = np.array_equal(results[0][0], results[1][0])
same_12 = np.array_equal(results[1][0], results[2][0])
print(f'\nTrial 1 == Trial 2: {same_01}')
print(f'Trial 2 == Trial 3: {same_12}')
print()
if not (same_01 and same_12):
    print('Results differ across runs! Set random_state=42 for reproducibility.')
else:
    print('Results happened to match, but this is NOT guaranteed without random_state.')

---

## Part 2: FIX — Correct the Pipeline

Now write the **correct** clustering pipeline from scratch, fixing all four errors:

1. **Standardize** features with `StandardScaler` BEFORE clustering
2. **Use `n_clusters=4`**, not `k=4`
3. **Apply PCA AFTER standardization**, not before
4. **Set `random_state=42`** for reproducibility

**Verification checkpoints:**
- Standardized features should have mean ~ 0, std ~ 1
- PCA on standardized data: PC1 should explain 35-50% of variance (NOT 90%+)
- Silhouette score for K=4 should be between 0.15 and 0.40
- Cluster sizes should be roughly balanced (not 1 giant cluster + 3 tiny ones)

In [ ]:
# -----------------------------------------------------------
# CORRECTED — Fixed clustering pipeline (all 4 errors resolved)
# -----------------------------------------------------------

# FIX 1: Standardize features BEFORE clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_names])

print("=== Verification: Standardization ===")
print(f"Feature means (should be ~0): {X_scaled.mean(axis=0).round(4)}")
print(f"Feature stds  (should be ~1): {X_scaled.std(axis=0).round(4)}")
print()

# FIX 2: Use n_clusters=4, not k=4
# FIX 4: Set random_state=42 for reproducibility
kmeans = KMeans(n_clusters=4, init='k-means++', n_init='auto', random_state=42)
labels = kmeans.fit_predict(X_scaled)

print("=== Cluster sizes ===")
print(np.bincount(labels))
print()

# FIX 3: Apply PCA AFTER standardization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("=== Verification: PCA on standardized data ===")
print(f"PC1 variance explained: {pca.explained_variance_ratio_[0]:.1%}")
print(f"PC2 variance explained: {pca.explained_variance_ratio_[1]:.1%}")
print("(Should be 35-50%, NOT 90%+ — that would indicate raw data was used)")
print()

sil = silhouette_score(X_scaled, labels)
print(f"Silhouette score (K=4): {sil:.4f}  (expected: 0.15 – 0.40)")
print()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels,
                           cmap='Set1', alpha=0.7, s=40, edgecolors='k', linewidths=0.3)
axes[0].set_title("PCA Projection — K-Means Clusters (K=4)", fontsize=12)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Cluster profile: mean of each feature per cluster
cluster_profiles = pd.DataFrame(X_scaled, columns=feature_names)
cluster_profiles['cluster'] = labels
means = cluster_profiles.groupby('cluster')[feature_names].mean()

im = axes[1].imshow(means.T, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels([f'C{k}' for k in range(4)])
axes[1].set_yticks(range(len(feature_names)))
axes[1].set_yticklabels(feature_names, fontsize=9)
axes[1].set_title("Cluster Feature Profiles (z-scores)", fontsize=12)
plt.colorbar(im, ax=axes[1], label='z-score')

plt.tight_layout()
plt.show()

# Country-level summary per cluster
print("=== Cluster Summaries (original scale) ===")
for k in range(4):
    mask = labels == k
    print(f"\nCluster {k} ({mask.sum()} countries):")
    print(f"  GDP/cap:       ${df.loc[mask, 'gdp_per_capita_ppp'].mean():>10,.0f}")
    print(f"  Life Exp:      {df.loc[mask, 'life_expectancy'].mean():>10.1f}")
    print(f"  Infant mort:   {df.loc[mask, 'infant_mortality'].mean():>10.1f}")
    print(f"  Internet (%):  {df.loc[mask, 'internet_users_pct'].mean():>10.1f}")


---

## Part 3: EXTEND — Customer Segmentation with Synthetic Data

Move beyond country-level data. In this section, you apply clustering to
a **customer segmentation** problem using synthetic behavioral data.
This mirrors how fintechs like Nubank (Chapter 22 opening hook) discover
customer archetypes from transaction patterns.

Then compare **PCA** and **UMAP** for dimensionality reduction.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Generate synthetic customer data with 4 latent segments
# -----------------------------------------------------------

from sklearn.datasets import make_blobs

np.random.seed(42)

# Create 4 customer segments with 6 behavioral features
n_customers = 2000
segment_centers = [
    [50, 5, 80, 10, 2, 30],    # Budget-conscious: low spend, few txns, high app usage
    [200, 20, 40, 50, 8, 70],   # Power users: high spend, many txns
    [120, 12, 60, 30, 5, 50],   # Moderate users
    [300, 30, 20, 80, 12, 90],  # Premium: very high spend, low app engagement
]

X_cust, y_true = make_blobs(
    n_samples=n_customers,
    centers=segment_centers,
    cluster_std=[15, 25, 20, 20],
    random_state=42
)

cust_features = [
    'avg_monthly_spend', 'txn_frequency', 'app_sessions',
    'credit_utilization', 'products_held', 'digital_engagement'
]

cust_df = pd.DataFrame(X_cust, columns=cust_features)
cust_df['true_segment'] = y_true

print(f'Customers: {len(cust_df)}')
print(f'Features: {cust_features}')
print(f'True segments: {cust_df["true_segment"].value_counts().sort_index().to_dict()}')
print()
print(cust_df[cust_features].describe().round(1))

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Cluster customers and compare PCA vs UMAP
# -----------------------------------------------------------

import umap

# Step 1: Standardize customer features
cust_scaler = StandardScaler()
X_cust_scaled = cust_scaler.fit_transform(cust_df[cust_features])

# Step 2: Fit K-Means with K=4 (we know the true number here)
km_cust = KMeans(n_clusters=4, init='k-means++', n_init='auto', random_state=42)
cust_df['kmeans_cluster'] = km_cust.fit_predict(X_cust_scaled)

# Step 3: PCA projection
pca_cust = PCA(n_components=2)
X_pca_cust = pca_cust.fit_transform(X_cust_scaled)

# Step 4: UMAP projection
# n_neighbors=15 balances local vs global structure (good default)
# min_dist=0.1 keeps cluster points tight together
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42
)
X_umap_cust = reducer.fit_transform(X_cust_scaled)

# Step 5: Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: PCA with TRUE labels
scatter1 = axes[0].scatter(X_pca_cust[:, 0], X_pca_cust[:, 1],
                           c=cust_df['true_segment'], cmap='Set1',
                           alpha=0.5, s=15)
axes[0].set_title('PCA — True Segments', fontsize=12)
axes[0].set_xlabel(f'PC1 ({pca_cust.explained_variance_ratio_[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({pca_cust.explained_variance_ratio_[1]:.1%} var)')
plt.colorbar(scatter1, ax=axes[0])

# Panel 2: PCA with K-MEANS labels
scatter2 = axes[1].scatter(X_pca_cust[:, 0], X_pca_cust[:, 1],
                           c=cust_df['kmeans_cluster'], cmap='Set1',
                           alpha=0.5, s=15)
axes[1].set_title('PCA — K-Means Clusters', fontsize=12)
axes[1].set_xlabel(f'PC1 ({pca_cust.explained_variance_ratio_[0]:.1%} var)')
plt.colorbar(scatter2, ax=axes[1])

# Panel 3: UMAP with K-MEANS labels
scatter3 = axes[2].scatter(X_umap_cust[:, 0], X_umap_cust[:, 1],
                           c=cust_df['kmeans_cluster'], cmap='Set1',
                           alpha=0.5, s=15)
axes[2].set_title('UMAP — K-Means Clusters', fontsize=12)
axes[2].set_xlabel('UMAP 1')
axes[2].set_ylabel('UMAP 2')
plt.colorbar(scatter3, ax=axes[2])

plt.tight_layout()
plt.show()

# Silhouette comparison
sil_kmeans = silhouette_score(X_cust_scaled, cust_df['kmeans_cluster'])
print(f'Silhouette score (K-Means K=4): {sil_kmeans:.4f}')
print()
print(f'PCA PC1+PC2 variance explained: {sum(pca_cust.explained_variance_ratio_[:2]):.1%}')
print()

# Agreement between K-Means clusters and true segments (cross-tab)
print("K-Means vs True Segments cross-tabulation:")
ct = pd.crosstab(cust_df['true_segment'], cust_df['kmeans_cluster'],
                  rownames=['True'], colnames=['K-Means'])
print(ct)
print()
print("Observation: UMAP separates clusters more cleanly than PCA when")
print("cluster boundaries are non-linear. PCA is linear; UMAP preserves")
print("local manifold structure, giving tighter, more separated blobs.")


---

## Part 4: Module Output — `clustering_utils.py`

Write a reusable Python module with three functions for clustering pipelines.
This is a **portfolio artifact** that demonstrates production-grade unsupervised learning.

### Requirements

```python
# clustering_utils.py

def run_kmeans_pipeline(df, features, k, random_state=42):
    """End-to-end K-Means pipeline: standardize, fit, return labels + metadata."""
    ...

def evaluate_k_range(X, k_range, random_state=42):
    """Compute WCSS and silhouette scores for a range of K values."""
    ...

def plot_pca_clusters(X, labels, feature_names):
    """PCA 2D scatter with cluster coloring + loadings annotation."""
    ...
```

In [ ]:
# -----------------------------------------------------------
# clustering_utils.py — Reusable Clustering Pipeline Module
# -----------------------------------------------------------

# %%writefile clustering_utils.py
"""
clustering_utils.py — Reusable Clustering Pipeline Module

Functions for standardized K-Means clustering, K evaluation,
and PCA visualization.

Author: ECON 5200 — Lab 22
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from typing import List, Dict


def run_kmeans_pipeline(
    df: pd.DataFrame,
    features: List[str],
    k: int,
    random_state: int = 42
) -> Dict:
    """End-to-end K-Means pipeline.

    1. Extracts features from DataFrame
    2. Standardizes with StandardScaler
    3. Fits K-Means
    4. Returns labels, scaler, model, and silhouette score

    Args:
        df: DataFrame with feature columns
        features: List of column names to use
        k: Number of clusters
        random_state: Random seed for reproducibility

    Returns:
        dict with keys: 'labels', 'scaler', 'model', 'X_scaled',
                        'silhouette', 'inertia'
    """
    X = df[features].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = KMeans(n_clusters=k, init='k-means++', n_init='auto', random_state=random_state)
    labels = model.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels) if k > 1 else float('nan')

    return {
        'labels':     labels,
        'scaler':     scaler,
        'model':      model,
        'X_scaled':   X_scaled,
        'silhouette': sil,
        'inertia':    model.inertia_,
    }


def evaluate_k_range(
    X: np.ndarray,
    k_range: range,
    random_state: int = 42
) -> pd.DataFrame:
    """Evaluate clustering quality across a range of K values.

    Computes WCSS (inertia) and silhouette score for each K.

    Args:
        X: Standardized feature matrix (already scaled)
        k_range: Range of K values to test (e.g., range(2, 11))
        random_state: Random seed

    Returns:
        DataFrame with columns: 'k', 'wcss', 'silhouette'
    """
    rows = []
    for k in k_range:
        km = KMeans(n_clusters=k, init='k-means++', n_init='auto', random_state=random_state)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels)
        rows.append({'k': k, 'wcss': km.inertia_, 'silhouette': sil})

    df_eval = pd.DataFrame(rows)

    # Plot elbow + silhouette side by side
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(df_eval['k'], df_eval['wcss'], 'o-', color='steelblue')
    axes[0].set_xlabel('Number of Clusters (K)')
    axes[0].set_ylabel('WCSS (Inertia)')
    axes[0].set_title('Elbow Curve')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df_eval['k'], df_eval['silhouette'], 'o-', color='darkorange')
    axes[1].set_xlabel('Number of Clusters (K)')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('Silhouette by K')
    axes[1].grid(True, alpha=0.3)

    best_k = df_eval.loc[df_eval['silhouette'].idxmax(), 'k']
    axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.6, label=f'Best K={best_k}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return df_eval


def plot_pca_clusters(
    X: np.ndarray,
    labels: np.ndarray,
    feature_names: List[str]
) -> None:
    """PCA 2D scatter plot with cluster coloring.

    Fits PCA(n_components=2), creates scatter plot colored by cluster,
    and annotates with explained variance ratios.

    Args:
        X: Standardized feature matrix
        labels: Cluster labels (array of integers)
        feature_names: List of original feature names
    """
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    n_clusters = len(np.unique(labels))
    cmap = plt.cm.get_cmap('Set1', n_clusters)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: scatter colored by cluster
    for k in range(n_clusters):
        mask = labels == k
        axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        label=f'Cluster {k} (n={mask.sum()})',
                        alpha=0.7, s=30, color=cmap(k), edgecolors='k', linewidths=0.2)

    axes[0].set_title('PCA Projection — Cluster Assignments', fontsize=12)
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
    axes[0].legend(loc='best', fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # Panel 2: top PCA loadings bar chart
    loadings_pc1 = pd.Series(pca.components_[0], index=feature_names).sort_values()
    loadings_pc1.plot(kind='barh', ax=axes[1], color=['firebrick' if v > 0 else 'steelblue' for v in loadings_pc1])
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_title('PC1 Feature Loadings', fontsize=12)
    axes[1].set_xlabel('Loading')
    axes[1].grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    plt.show()

    total_var = sum(pca.explained_variance_ratio_[:2])
    print(f"PC1+PC2 explain {total_var:.1%} of total variance.")


# --- Quick self-test ---
if __name__ == '__main__':
    from sklearn.datasets import make_blobs
    X_test, _ = make_blobs(n_samples=200, centers=3, n_features=5, random_state=0)
    df_test = pd.DataFrame(X_test, columns=[f'f{i}' for i in range(5)])

    result = run_kmeans_pipeline(df_test, [f'f{i}' for i in range(5)], k=3)
    print(f'Labels shape: {result["labels"].shape}')
    print(f'Silhouette:   {result["silhouette"]:.4f}')

    eval_df = evaluate_k_range(result['X_scaled'], range(2, 8))
    print(eval_df)

    plot_pca_clusters(result['X_scaled'], result['labels'], [f'f{i}' for i in range(5)])
    print('Self-test passed.')


---

## Challenge: Hierarchical Clustering Comparison

K-Means assumes spherical clusters and requires you to specify K upfront.
**Agglomerative hierarchical clustering** builds a tree (dendrogram) of
nested clusters and lets you choose K after inspecting the tree.

Compare K-Means and Agglomerative clustering on the WDI data:
1. Fit `AgglomerativeClustering(n_clusters=4)` on the standardized WDI data
2. Plot the dendrogram using `scipy.cluster.hierarchy`
3. Compare cluster assignments with K-Means — do they agree?

In [ ]:
# -----------------------------------------------------------
# CHALLENGE — Hierarchical clustering + dendrogram
# -----------------------------------------------------------

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Reuse the standardized WDI data from Part 2
# (X_scaled and labels are already defined above)

# Step 1: Fit Agglomerative Clustering
agglo = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_agglo = agglo.fit_predict(X_scaled)

print("=== Agglomerative Clustering (Ward linkage, K=4) ===")
print(f"Cluster sizes: {np.bincount(labels_agglo)}")
sil_agglo = silhouette_score(X_scaled, labels_agglo)
print(f"Silhouette score: {sil_agglo:.4f}")
print()

# Step 2: Dendrogram (Ward linkage on full dataset)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

linkage_matrix = linkage(X_scaled, method='ward')
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=axes[0]
)
axes[0].set_title("Dendrogram (Ward linkage, last 30 merges)", fontsize=11)
axes[0].set_xlabel("Country (or merged cluster size)")
axes[0].set_ylabel("Ward Distance")
# Mark the 4-cluster cut level
cut_level = linkage_matrix[-(4-1), 2]
axes[0].axhline(y=cut_level, color='red', linestyle='--', alpha=0.7, label=f'K=4 cut ({cut_level:.1f})')
axes[0].legend()

# Step 3: Cross-tabulate K-Means vs Agglomerative
ct = pd.crosstab(labels, labels_agglo,
                  rownames=['K-Means'], colnames=['Agglomerative'])
print("Cross-tabulation: K-Means vs Agglomerative cluster assignments")
print(ct)
print()

# Heatmap of the cross-tab
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title("K-Means vs Agglomerative\n(cell = # countries in both clusters)", fontsize=11)
axes[1].set_xlabel("Agglomerative Cluster")
axes[1].set_ylabel("K-Means Cluster")

plt.tight_layout()
plt.show()

# Step 4: Silhouette comparison
sil_kmeans_ref = silhouette_score(X_scaled, labels)
print(f"Silhouette — K-Means (K=4):        {sil_kmeans_ref:.4f}")
print(f"Silhouette — Agglomerative (K=4):  {sil_agglo:.4f}")
print()

agreement = np.mean(
    [np.max([np.mean(labels_agglo[labels == k] == j) for j in range(4)])
     for k in range(4)]
)
print(f"Cluster assignment agreement (approx): {agreement:.1%}")
print()
print("Observation:")
print("- K-Means optimizes WCSS (spherical clusters); Agglomerative uses")
print("  hierarchical merging — neither is universally better.")
print("- High cross-tab agreement on diagonal indicates both methods")
print("  discover similar structure in the data.")
print("- Dendrogram helps choose K visually: look for the largest")
print("  vertical gap before a merge (the 'elbow' in the tree).")


---

## Digital Portfolio: Institutional Signaling

### Generate Your Professional README

Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

```text
"I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed and fixed a broken K-Means pipeline (missing standardization,
  wrong parameter name, PCA before scaling, no random_state)
* Built a corrected pipeline: StandardScaler -> K-Means -> PCA visualization
* Applied clustering to customer segmentation with synthetic behavioral data
* Compared PCA vs UMAP for dimensionality reduction
* Built a reusable clustering_utils.py module with run_kmeans_pipeline(),
  evaluate_k_range(), and plot_pca_clusters()
* Key finding: [FILL IN — what K was optimal? How did PCA vs UMAP compare?]

**Please write a README.md entry including:**
1. Project Title: Unsupervised Learning — Clustering & Dimensionality Reduction
2. Objective: A professional one-sentence summary
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Make this sound like a professional tech economist wrote it."
```

### Push to GitHub

```bash
cd econ-lab-22-clustering
git add notebooks/ src/ figures/ README.md
git commit -m "Lab 22: Clustering Economies — K-Means, PCA, UMAP & Module"
git push origin main
```

Submit your GitHub repo link on Canvas.